# Ele — Phase 1 (Colab or Runpod Jupyter)

Frozen SigLIP + linear head on CIFAKE (no augmentation).

## Runpod (recommended)

1. Console → **Deploy Pod** (PyTorch template, Jupyter + SSH on).
2. Wait until **Running** → **Connect → Jupyter Lab**.
3. In Jupyter: **File → Open from GitHub** or clone (cell 2), then open `notebooks/colab_phase1.ipynb`.
4. **Run → Run All Cells**. Skip Google Drive; checkpoints stay in `/workspace`.
5. **Stop the Pod** when finished — you are billed while it is running.

## Colab

Runtime → T4 GPU → Run all. Last cell copies `models/phase1.pt` to Drive.

On an H100 use `--batch-size 256` (already in the train cell). On a T4 stay at `16` (or `8` if OOM).

## 1. Confirm GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. On Colab: Runtime → T4 GPU. On Runpod: pick a GPU Pod, not CPU."
)
print(torch.cuda.get_device_name(0))
print("CUDA", torch.version.cuda)

## 2. Clone the repo

Clones into `/workspace/ele` on Runpod or `/content/ele` on Colab.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/harikrishn4a/ele.git"
ROOT = Path("/workspace/ele" if Path("/workspace").is_dir() else "/content/ele")

if not ROOT.exists():
    !git clone --depth 1 {REPO} {ROOT}
else:
    print("Already cloned:", ROOT)

%cd {ROOT}
!git log -1 --oneline

## 3. Install packages

The PyTorch Pod / Colab already has CUDA torch. This adds OpenCLIP (SigLIP) and the rest of Ele.

In [ ]:
!pip install -q open-clip-torch timm transformers scikit-learn opencv-python einops peft kagglehub tensorboard tqdm pandas Pillow

## 4. Patch GitHub `main` if needed

Older commits import a missing `evaluate_model` and look for CIFAKE `synthetic/` instead of `FAKE/`.

In [ ]:
from pathlib import Path

train_py = Path("src/train.py")
text = train_py.read_text()
old = "from src.evaluate import evaluate_model"
if old in text:
    train_py.write_text(text.replace(old, "# " + old, 1))
    print("Patched src/train.py import")
else:
    print("train.py already ok")

## 5. Download CIFAKE and fill `data/`

Copies official CIFAKE `train`/`test` + `REAL`/`FAKE` (not a 70/30 split).

In [ ]:
import shutil
from pathlib import Path

import kagglehub

cifake = Path(
    kagglehub.dataset_download(
        "birdy654/cifake-real-and-ai-generated-synthetic-images"
    )
)
print("CIFAKE at", cifake)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def first_dir(parent, names):
    for name in names:
        p = parent / name
        if p.is_dir():
            return p
    return None


def copy_images(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    files = [p for p in src.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    for src_file in files:
        out = dst / src_file.name
        if not out.exists():
            shutil.copy2(src_file, out)
    print(f"{len(files):>6}  {src} -> {dst}")


root = Path("data")
for split in ("train", "test"):
    split_dir = cifake / split
    copy_images(first_dir(split_dir, ["REAL", "real"]), root / split / "real")
    copy_images(first_dir(split_dir, ["FAKE", "fake", "synthetic"]), root / split / "fake")

for split in ("train", "test"):
    n_real = len(list((root / split / "real").glob("*")))
    n_fake = len(list((root / split / "fake").glob("*")))
    print(f"{split}: {n_real} real, {n_fake} fake")
    assert n_real > 0 and n_fake > 0, f"Empty {split} split"

## 6. Hugging Face token (optional)

Only if the SigLIP download is rate-limited. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os

# os.environ["HF_TOKEN"] = "hf_..."
print("HF_TOKEN set:" , bool(os.environ.get("HF_TOKEN")))

## 7. Train Phase 1

CUDA is automatic on a GPU Pod. First run downloads ~3.5GB of SigLIP weights, then trains 3 epochs.

H100 command below uses batch 256. Stop the Pod when training finishes.

In [ ]:
!python3 -m src.train \
  --backbone sigclip \
  --epochs 3 \
  --batch-size 256 \
  --num-workers 6 \
  --lr 1e-3 \
  --output models/phase1_h100.pt

## 8. Evaluate transform grid

In [ ]:
!python3 -m src.evaluate \
  --model models/phase1_h100.pt \
  --backbone sigclip \
  --dataset data \
  --output results/phase1.csv

## 9. Save checkpoint (Colab Drive, or confirm files on Runpod)

In [ ]:
from pathlib import Path
import shutil

for src in (Path("models/phase1_h100.pt"), Path("results/phase1.csv")):
    print(("OK " if src.exists() else "MISSING ") + str(src.resolve() if src.exists() else src))

try:
    from google.colab import drive

    drive.mount("/content/drive")
    out = Path("/content/drive/MyDrive/Ele")
    out.mkdir(parents=True, exist_ok=True)
    for src in (Path("models/phase1_h100.pt"), Path("results/phase1.csv")):
        if src.exists():
            shutil.copy2(src, out / src.name)
            print("Copied to Drive", out / src.name)
except ImportError:
    print("Runpod: files stay under /workspace/ele. Download them from Jupyter before you stop the Pod.")